# Lesson 1A: K-Means Clustering — Theory and Mathematical Foundations

---

## Overview

**K-Means** is one of the most widely used unsupervised learning algorithms for partitioning data into K distinct, non-overlapping clusters. It is a centroid-based algorithm that iteratively assigns data points to clusters and updates cluster centers to minimize within-cluster variance.

**Historical Context**: The K-Means algorithm was first proposed by Stuart Lloyd in 1957 (published in 1982) and independently discovered by J. MacQueen in 1967.

**Key References**:
- Lloyd, S. (1982). "Least Squares Quantization in PCM". *IEEE Transactions on Information Theory*, 28(2), 129-137.
- MacQueen, J. (1967). "Some Methods for Classification and Analysis of Multivariate Observations". *Proceedings of the Fifth Berkeley Symposium on Mathematical Statistics and Probability*, 1, 281-297.
- Arthur, D., & Vassilvitskii, S. (2007). "k-means++: The Advantages of Careful Seeding". *Proceedings of the Eighteenth Annual ACM-SIAM Symposium on Discrete Algorithms*, 1027-1035.

---

## Table of Contents

1. Problem Formulation
2. Mathematical Objective
3. Algorithm Description
4. Convergence Analysis
5. Computational Complexity
6. Initialization Methods
7. Practical Implementation
8. Limitations and Extensions

In [ ]:
# Import required libraries
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans
import seaborn as sns

# Set visualization parameters
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

# Reproducibility
np.random.seed(42)

print("Environment configured successfully")
print(f"NumPy version: {np.__version__}")

---

## 1. Problem Formulation

### 1.1 Clustering Objective

Given a dataset $\mathcal{D} = \{\mathbf{x}_1, \mathbf{x}_2, \ldots, \mathbf{x}_n\}$ where $\mathbf{x}_i \in \mathbb{R}^d$, the goal is to partition the data into $K$ clusters $\mathcal{C} = \{C_1, C_2, \ldots, C_K\}$ such that:

1. **Homogeneity**: Points within the same cluster are similar
2. **Separation**: Points in different clusters are dissimilar
3. **Compactness**: Clusters have minimal within-cluster variance

### 1.2 Formal Definition

A partition $\mathcal{C}$ is defined by a set of cluster assignments:

$$z_i \in \{1, 2, \ldots, K\}, \quad i = 1, \ldots, n$$

where $z_i = k$ indicates that point $\mathbf{x}_i$ belongs to cluster $C_k$.

Each cluster $C_k$ is represented by its centroid (mean):

$$\boldsymbol{\mu}_k = \frac{1}{|C_k|} \sum_{\mathbf{x}_i \in C_k} \mathbf{x}_i$$

where $|C_k|$ is the number of points in cluster $C_k$.

---

## 2. Mathematical Objective

### 2.1 Within-Cluster Sum of Squares (WCSS)

K-Means aims to minimize the **within-cluster sum of squares** (also called inertia):

$$J(\mathcal{C}, \boldsymbol{\mu}) = \sum_{k=1}^{K} \sum_{\mathbf{x}_i \in C_k} \|\mathbf{x}_i - \boldsymbol{\mu}_k\|^2$$

This can be rewritten using indicator variables $r_{ik} \in \{0, 1\}$ where $r_{ik} = 1$ if $\mathbf{x}_i \in C_k$:

$$J = \sum_{i=1}^{n} \sum_{k=1}^{K} r_{ik} \|\mathbf{x}_i - \boldsymbol{\mu}_k\|^2$$

### 2.2 Optimization Problem

The K-Means objective is a **non-convex optimization problem**:

$$\min_{\mathcal{C}, \boldsymbol{\mu}} \sum_{k=1}^{K} \sum_{\mathbf{x}_i \in C_k} \|\mathbf{x}_i - \boldsymbol{\mu}_k\|^2$$

**Challenges**:
- Non-convex: Multiple local minima exist
- Combinatorial: $K^n$ possible assignments (intractable for large $n$)
- NP-hard in general case

**Solution Approach**: Use iterative coordinate descent (Lloyd's algorithm)

### 2.3 Relationship to Gaussian Mixture Models

K-Means can be viewed as a special case of Expectation-Maximization (EM) for Gaussian Mixture Models with:
- Equal, spherical covariances: $\Sigma_k = \sigma^2 \mathbf{I}$
- As $\sigma \to 0$, soft assignments become hard assignments

---

## 3. Algorithm Description

### 3.1 Lloyd's Algorithm (Standard K-Means)

**Input**: Dataset $\mathcal{D} = \{\mathbf{x}_1, \ldots, \mathbf{x}_n\}$, number of clusters $K$

**Output**: Cluster assignments $\{z_1, \ldots, z_n\}$, centroids $\{\boldsymbol{\mu}_1, \ldots, \boldsymbol{\mu}_K\}$

**Algorithm**:

```
1. Initialize centroids μ₁, ..., μ_K (random selection from data)

2. Repeat until convergence:
   
   a. Assignment Step:
      For each data point xᵢ:
          zᵢ ← argmin_k ||xᵢ - μ_k||²
   
   b. Update Step:
      For each cluster k:
          μ_k ← (1/|C_k|) Σ_{xᵢ ∈ C_k} xᵢ

3. Return assignments z and centroids μ
```

**Convergence Criterion**: 
- Centroids do not change: $\|\boldsymbol{\mu}_k^{(t+1)} - \boldsymbol{\mu}_k^{(t)}\| < \epsilon$
- Or maximum iterations reached

### 3.2 Coordinate Descent Interpretation

The algorithm alternates between two steps:

1. **E-step** (Assignment): Fix $\boldsymbol{\mu}$, optimize assignments
   $$z_i^{(t+1)} = \arg\min_k \|\mathbf{x}_i - \boldsymbol{\mu}_k^{(t)}\|^2$$

2. **M-step** (Update): Fix assignments, optimize centroids
   $$\boldsymbol{\mu}_k^{(t+1)} = \arg\min_{\boldsymbol{\mu}} \sum_{i: z_i = k} \|\mathbf{x}_i - \boldsymbol{\mu}\|^2$$

The M-step has a closed-form solution: the sample mean.

In [ ]:
# Implement K-Means from scratch for educational purposes

class KMeansFromScratch:
    """
    K-Means clustering implementation using Lloyd's algorithm.
    
    This is an educational implementation. For production use sklearn.cluster.KMeans.
    """
    
    def __init__(self, n_clusters=3, max_iter=300, tol=1e-4, random_state=None):
        """
        Parameters
        ----------
        n_clusters : int
            Number of clusters K
        max_iter : int
            Maximum number of iterations
        tol : float
            Convergence tolerance (change in centroids)
        random_state : int or None
            Random seed for reproducibility
        """
        self.n_clusters = n_clusters
        self.max_iter = max_iter
        self.tol = tol
        self.random_state = random_state
        
        # Will be set during fit
        self.centroids = None
        self.labels_ = None
        self.inertia_ = None
        self.n_iter_ = 0
        
    def _initialize_centroids(self, X):
        """
        Initialize centroids by randomly selecting K points from data.
        
        This is the 'forgy' initialization method.
        """
        n_samples = X.shape[0]
        
        if self.random_state is not None:
            np.random.seed(self.random_state)
        
        # Randomly select K data points as initial centroids
        indices = np.random.choice(n_samples, self.n_clusters, replace=False)
        return X[indices].copy()
    
    def _assign_clusters(self, X, centroids):
        """
        Assignment step: Assign each point to nearest centroid.
        
        Returns
        -------
        labels : array of shape (n_samples,)
            Cluster assignment for each point
        """
        # Compute distances to all centroids
        # distances[i, k] = ||x_i - μ_k||²
        distances = np.zeros((X.shape[0], self.n_clusters))
        
        for k in range(self.n_clusters):
            distances[:, k] = np.linalg.norm(X - centroids[k], axis=1) ** 2
        
        # Assign to nearest centroid
        labels = np.argmin(distances, axis=1)
        
        return labels
    
    def _update_centroids(self, X, labels):
        """
        Update step: Recompute centroids as cluster means.
        
        Returns
        -------
        centroids : array of shape (n_clusters, n_features)
            Updated cluster centroids
        """
        centroids = np.zeros((self.n_clusters, X.shape[1]))
        
        for k in range(self.n_clusters):
            # Find all points in cluster k
            cluster_points = X[labels == k]
            
            if len(cluster_points) > 0:
                # Compute mean
                centroids[k] = cluster_points.mean(axis=0)
            else:
                # Handle empty cluster: reinitialize randomly
                centroids[k] = X[np.random.randint(X.shape[0])]
        
        return centroids
    
    def _compute_inertia(self, X, labels, centroids):
        """
        Compute within-cluster sum of squares (WCSS).
        
        Inertia = Σ_k Σ_{x_i ∈ C_k} ||x_i - μ_k||²
        """
        inertia = 0.0
        
        for k in range(self.n_clusters):
            cluster_points = X[labels == k]
            if len(cluster_points) > 0:
                inertia += np.sum((cluster_points - centroids[k]) ** 2)
        
        return inertia
    
    def fit(self, X):
        """
        Fit K-Means clustering.
        
        Parameters
        ----------
        X : array of shape (n_samples, n_features)
            Training data
        
        Returns
        -------
        self : object
            Fitted estimator
        """
        # Initialize centroids
        self.centroids = self._initialize_centroids(X)
        
        # Iterative optimization
        for iteration in range(self.max_iter):
            # E-step: Assign points to clusters
            labels = self._assign_clusters(X, self.centroids)
            
            # M-step: Update centroids
            new_centroids = self._update_centroids(X, labels)
            
            # Check convergence
            centroid_shift = np.linalg.norm(new_centroids - self.centroids)
            
            self.centroids = new_centroids
            self.n_iter_ = iteration + 1
            
            if centroid_shift < self.tol:
                break
        
        # Final assignment and inertia
        self.labels_ = self._assign_clusters(X, self.centroids)
        self.inertia_ = self._compute_inertia(X, self.labels_, self.centroids)
        
        return self
    
    def predict(self, X):
        """
        Predict cluster labels for new data.
        
        Parameters
        ----------
        X : array of shape (n_samples, n_features)
            New data
        
        Returns
        -------
        labels : array of shape (n_samples,)
            Cluster assignments
        """
        if self.centroids is None:
            raise ValueError("Model must be fitted before prediction")
        
        return self._assign_clusters(X, self.centroids)

# Test the implementation
print("K-Means implementation complete")
print("\nTesting on synthetic data...")

# Generate synthetic dataset
X_test, y_true = make_blobs(n_samples=300, centers=3, n_features=2, 
                            cluster_std=0.6, random_state=42)

# Fit K-Means
kmeans = KMeansFromScratch(n_clusters=3, random_state=42)
kmeans.fit(X_test)

print(f"\nConverged in {kmeans.n_iter_} iterations")
print(f"Final inertia (WCSS): {kmeans.inertia_:.2f}")
print(f"Centroids:\n{kmeans.centroids}")

---

## 4. Convergence Analysis

### 4.1 Theoretical Guarantees

**Theorem 1** (Monotonic Decrease): Each iteration of K-Means decreases or maintains the objective function:

$$J^{(t+1)} \leq J^{(t)}$$

**Proof Sketch**:
- Assignment step: By definition, assigns each point to nearest centroid, minimizing distance
- Update step: Centroid is optimal given fixed assignments (mean minimizes squared distances)
- Therefore: $J$ is non-increasing

**Theorem 2** (Finite Convergence): Lloyd's algorithm converges in a finite number of steps.

**Proof**: 
- Only finitely many possible partitions ($\leq K^n$)
- Objective strictly decreases unless at local minimum
- Cannot revisit same partition (would require $J$ to increase)
- Therefore must reach local minimum in finite steps

### 4.2 Convergence Rate

**Empirical Observation**: K-Means often converges quickly in practice:
- Typically < 100 iterations for well-separated clusters
- Most improvement in first 10-20 iterations

**Theoretical Result** (Bottou & Bengio, 1995):
- Convergence is exponentially fast for well-separated clusters
- Requires separation condition: $\min_{k \neq k'} \|\boldsymbol{\mu}_k - \boldsymbol{\mu}_{k'}\| > c \cdot \sigma$

### 4.3 Quality of Solution

**Important**: K-Means only guarantees convergence to a **local minimum**, not global minimum.

**Factors affecting solution quality**:
1. Initialization (see Section 6)
2. Data distribution (works best for spherical, well-separated clusters)
3. Number of clusters $K$

---

## 5. Computational Complexity

### 5.1 Time Complexity

**Per iteration**:
- **Assignment step**: $O(nKd)$
  - For each of $n$ points, compute distance to $K$ centroids in $d$ dimensions
- **Update step**: $O(nd)$
  - Sum $n$ points in $d$ dimensions, divide by cluster sizes

**Total**: $O(t \cdot n \cdot K \cdot d)$

where $t$ is the number of iterations.

**Typical values**:
- $t \approx 10-100$ iterations
- For $n = 10^6$, $K = 100$, $d = 100$: ~1-10 billion operations

### 5.2 Space Complexity

**Storage requirements**:
- Data: $O(nd)$ (input)
- Centroids: $O(Kd)$ (negligible compared to data)
- Labels: $O(n)$

**Total**: $O(nd + n + Kd) = O(nd)$

### 5.3 Scalability Improvements

**Mini-Batch K-Means** (Sculley, 2010):
- Use random subsamples (mini-batches) per iteration
- Complexity: $O(t \cdot b \cdot K \cdot d)$ where $b \ll n$
- Trade-off: Faster but less accurate

**Parallel/Distributed**:
- Assignment step is embarrassingly parallel
- Update step requires aggregation
- Implementations: Spark MLlib, Dask-ML

---

## 6. Initialization Methods

### 6.1 Random Initialization (Forgy Method)

**Procedure**: Select $K$ data points uniformly at random as initial centroids.

**Advantages**:
- Simple, fast
- No preprocessing required

**Disadvantages**:
- High variance in results
- May converge to poor local minima

### 6.2 K-Means++ Initialization

**Reference**: Arthur, D., & Vassilvitskii, S. (2007)

**Procedure**:
1. Choose first centroid $\boldsymbol{\mu}_1$ uniformly at random from data
2. For each subsequent centroid $k = 2, \ldots, K$:
   - Compute $D(\mathbf{x})^2 = \min_{j < k} \|\mathbf{x} - \boldsymbol{\mu}_j\|^2$ for all points
   - Choose $\boldsymbol{\mu}_k$ with probability $\propto D(\mathbf{x})^2$

**Advantages**:
- Provably better initialization
- Expected approximation ratio: $O(\log K)$ competitive with optimal

**Theorem** (Arthur & Vassilvitskii, 2007): 
$$\mathbb{E}[J_{\text{K-Means++}}] \leq 8(\ln K + 2) \cdot J_{\text{OPT}}$$

### 6.3 Multiple Random Restarts

**Strategy**: Run K-Means multiple times with different random initializations, keep best result.

**Implementation**:
```python
sklearn.cluster.KMeans(n_init=10)  # 10 random initializations
```

**Trade-off**: $10\times$ more computation, but more stable results.

In [ ]:
# Visualize K-Means algorithm execution

def visualize_kmeans_iterations(X, true_labels, n_clusters=3, random_state=42):
    """
    Visualize K-Means convergence over iterations.
    """
    kmeans = KMeansFromScratch(n_clusters=n_clusters, max_iter=1, random_state=random_state)
    
    # Run for multiple iterations and capture state
    max_plots = 6
    iterations_to_plot = [0, 1, 2, 3, 5, 10]
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    axes = axes.flatten()
    
    # Initialize
    kmeans.centroids = kmeans._initialize_centroids(X)
    
    for idx, target_iter in enumerate(iterations_to_plot):
        if idx == 0:
            # Initial state
            labels = kmeans._assign_clusters(X, kmeans.centroids)
            centroids = kmeans.centroids
            inertia = kmeans._compute_inertia(X, labels, centroids)
            title = f"Iteration 0 (Initialization)\nInertia: {inertia:.2f}"
        else:
            # Run iterations
            while kmeans.n_iter_ < target_iter:
                labels = kmeans._assign_clusters(X, kmeans.centroids)
                kmeans.centroids = kmeans._update_centroids(X, labels)
                kmeans.n_iter_ += 1
            
            inertia = kmeans._compute_inertia(X, labels, kmeans.centroids)
            title = f"Iteration {target_iter}\nInertia: {inertia:.2f}"
            centroids = kmeans.centroids
        
        # Plot
        ax = axes[idx]
        scatter = ax.scatter(X[:, 0], X[:, 1], c=labels, cmap='viridis', 
                           s=50, alpha=0.6, edgecolors='black', linewidths=0.5)
        ax.scatter(centroids[:, 0], centroids[:, 1], c='red', s=300, 
                  marker='X', edgecolors='black', linewidths=2, label='Centroids')
        ax.set_title(title, fontsize=13, fontweight='bold')
        ax.set_xlabel('Feature 1', fontsize=11)
        ax.set_ylabel('Feature 2', fontsize=11)
        ax.legend(loc='upper right')
        ax.grid(True, alpha=0.3)
    
    plt.suptitle('K-Means Algorithm Convergence', fontsize=16, fontweight='bold', y=1.00)
    plt.tight_layout()
    plt.show()

# Generate well-separated clusters
X_viz, y_viz = make_blobs(n_samples=300, centers=3, n_features=2,
                          cluster_std=0.8, random_state=42)

print("Visualizing K-Means convergence...")
visualize_kmeans_iterations(X_viz, y_viz, n_clusters=3, random_state=42)

---

## 7. Practical Implementation

### 7.1 Using Scikit-Learn

**Production-quality implementation** with optimizations:

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score

# Generate dataset
X, y_true = make_blobs(n_samples=500, centers=4, n_features=2,
                       cluster_std=1.0, random_state=42)

# Fit K-Means with best practices
kmeans_sklearn = KMeans(
    n_clusters=4,
    init='k-means++',      # Use k-means++ initialization
    n_init=10,             # Run 10 times with different initializations
    max_iter=300,          # Maximum iterations
    tol=1e-4,              # Convergence tolerance
    random_state=42,       # Reproducibility
    algorithm='lloyd'      # Lloyd's algorithm (default)
)

# Fit model
kmeans_sklearn.fit(X)

# Evaluation metrics
silhouette = silhouette_score(X, kmeans_sklearn.labels_)
davies_bouldin = davies_bouldin_score(X, kmeans_sklearn.labels_)

print("Scikit-Learn K-Means Results")
print("=" * 50)
print(f"Number of iterations: {kmeans_sklearn.n_iter_}")
print(f"Inertia (WCSS): {kmeans_sklearn.inertia_:.2f}")
print(f"\nEvaluation Metrics:")
print(f"  Silhouette Score: {silhouette:.3f} (range: [-1, 1], higher is better)")
print(f"  Davies-Bouldin Index: {davies_bouldin:.3f} (lower is better)")

# Visualize results
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# True labels
ax = axes[0]
scatter = ax.scatter(X[:, 0], X[:, 1], c=y_true, cmap='Set2', 
                    s=60, alpha=0.7, edgecolors='black', linewidths=0.5)
ax.set_title('True Cluster Labels', fontsize=14, fontweight='bold')
ax.set_xlabel('Feature 1', fontsize=12)
ax.set_ylabel('Feature 2', fontsize=12)
ax.grid(True, alpha=0.3)
plt.colorbar(scatter, ax=ax, label='True Cluster')

# Predicted labels
ax = axes[1]
scatter = ax.scatter(X[:, 0], X[:, 1], c=kmeans_sklearn.labels_, cmap='Set2',
                    s=60, alpha=0.7, edgecolors='black', linewidths=0.5)
ax.scatter(kmeans_sklearn.cluster_centers_[:, 0], 
          kmeans_sklearn.cluster_centers_[:, 1],
          c='red', s=400, marker='X', edgecolors='black', linewidths=2,
          label='Centroids', zorder=5)
ax.set_title(f'K-Means Clustering (K={4})', fontsize=14, fontweight='bold')
ax.set_xlabel('Feature 1', fontsize=12)
ax.set_ylabel('Feature 2', fontsize=12)
ax.legend(loc='upper right', fontsize=11)
ax.grid(True, alpha=0.3)
plt.colorbar(scatter, ax=ax, label='Predicted Cluster')

plt.tight_layout()
plt.show()

### 7.2 Determining Optimal K

**Elbow Method**: Plot inertia vs. K, look for "elbow" where marginal improvement decreases.

In [ ]:
# Elbow method for optimal K selection

K_range = range(2, 11)
inertias = []
silhouettes = []

for k in K_range:
    kmeans = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    kmeans.fit(X)
    inertias.append(kmeans.inertia_)
    silhouettes.append(silhouette_score(X, kmeans.labels_))

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Elbow curve
ax = axes[0]
ax.plot(K_range, inertias, 'bo-', linewidth=2, markersize=10)
ax.set_xlabel('Number of Clusters (K)', fontsize=12)
ax.set_ylabel('Inertia (WCSS)', fontsize=12)
ax.set_title('Elbow Method for Optimal K', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)

# Annotate elbow
elbow_k = 4  # True number of clusters
ax.annotate('Elbow Point', xy=(elbow_k, inertias[elbow_k-2]), 
           xytext=(elbow_k+1, inertias[elbow_k-2] + 500),
           arrowprops=dict(arrowstyle='->', lw=2, color='red'),
           fontsize=11, fontweight='bold', color='red')

# Silhouette scores
ax = axes[1]
ax.plot(K_range, silhouettes, 'go-', linewidth=2, markersize=10)
ax.set_xlabel('Number of Clusters (K)', fontsize=12)
ax.set_ylabel('Silhouette Score', fontsize=12)
ax.set_title('Silhouette Analysis', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.axhline(y=max(silhouettes), color='red', linestyle='--', linewidth=1,
          label=f'Maximum at K={K_range[np.argmax(silhouettes)]}')
ax.legend()

plt.tight_layout()
plt.show()

print(f"Elbow method suggests K ≈ 4")
print(f"Silhouette score maximized at K = {K_range[np.argmax(silhouettes)]}")

---

## 8. Limitations and Extensions

### 8.1 Limitations of K-Means

1. **Assumes spherical clusters**: Struggles with elongated or irregular shapes
2. **Sensitive to K**: Must specify number of clusters a priori
3. **Sensitive to initialization**: Different runs can yield different results
4. **Sensitive to outliers**: Outliers can distort centroids
5. **Assumes equal variance**: Poor performance with clusters of different sizes/densities
6. **Only finds convex clusters**: Cannot separate concentric circles

### 8.2 When K-Means Fails

**Example**: Non-spherical clusters

In [ ]:
# Demonstrate K-Means limitations

from sklearn.datasets import make_moons, make_circles

fig, axes = plt.subplots(2, 3, figsize=(18, 12))

datasets = [
    ('Moons (Non-convex)', make_moons(n_samples=300, noise=0.05, random_state=42)),
    ('Circles (Nested)', make_circles(n_samples=300, noise=0.05, factor=0.5, random_state=42)),
    ('Anisotropic (Elongated)', (np.dot(np.random.randn(300, 2), [[0.6, -0.6], [-0.4, 0.8]]), 
                                np.random.randint(0, 2, 300)))
]

for idx, (name, (X_data, y_data)) in enumerate(datasets):
    # True clusters
    ax = axes[idx, 0]
    ax.scatter(X_data[:, 0], X_data[:, 1], c=y_data, cmap='Set1', s=50, alpha=0.7,
              edgecolors='black', linewidths=0.5)
    ax.set_title(f'{name}\nTrue Labels', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3)
    
    # K-Means (K=2)
    kmeans = KMeans(n_clusters=2, random_state=42)
    labels_kmeans = kmeans.fit_predict(X_data)
    
    ax = axes[idx, 1]
    ax.scatter(X_data[:, 0], X_data[:, 1], c=labels_kmeans, cmap='Set1', s=50, alpha=0.7,
              edgecolors='black', linewidths=0.5)
    ax.scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1],
              c='red', s=300, marker='X', edgecolors='black', linewidths=2)
    ax.set_title('K-Means Clustering', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3)
    
    # Better alternative (DBSCAN for moons/circles, GMM for anisotropic)
    if idx < 2:
        from sklearn.cluster import DBSCAN
        dbscan = DBSCAN(eps=0.3, min_samples=5)
        labels_alt = dbscan.fit_predict(X_data)
        title = 'DBSCAN (Better)'
    else:
        from sklearn.mixture import GaussianMixture
        gmm = GaussianMixture(n_components=2, covariance_type='full', random_state=42)
        labels_alt = gmm.fit_predict(X_data)
        title = 'GMM (Better)'
    
    ax = axes[idx, 2]
    ax.scatter(X_data[:, 0], X_data[:, 1], c=labels_alt, cmap='Set1', s=50, alpha=0.7,
              edgecolors='black', linewidths=0.5)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3)

plt.suptitle('K-Means Limitations and Alternatives', fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()

print("K-Means fails on non-spherical, nested, or elongated clusters")
print("Consider DBSCAN (density-based) or GMM (flexible covariance) as alternatives")

### 8.3 Extensions and Variants

1. **K-Medoids (PAM)**: Use actual data points as centers (robust to outliers)

2. **Fuzzy C-Means**: Soft cluster assignments with membership degrees

3. **Gaussian Mixture Models**: Probabilistic clustering with flexible covariance

4. **K-Means++**: Better initialization (already discussed)

5. **Mini-Batch K-Means**: Scalable variant using subsamples

6. **Bisecting K-Means**: Hierarchical divisive approach

7. **Spherical K-Means**: For directional data (cosine distance)

---

## References

### Foundational Papers

1. **Lloyd, S.** (1982). "Least Squares Quantization in PCM". *IEEE Transactions on Information Theory*, 28(2), 129-137.

2. **MacQueen, J.** (1967). "Some Methods for Classification and Analysis of Multivariate Observations". *Proceedings of the Fifth Berkeley Symposium on Mathematical Statistics and Probability*, 1, 281-297.

3. **Arthur, D., & Vassilvitskii, S.** (2007). "k-means++: The Advantages of Careful Seeding". *Proceedings of the Eighteenth Annual ACM-SIAM Symposium on Discrete Algorithms*, 1027-1035.

### Theoretical Analysis

4. **Bottou, L., & Bengio, Y.** (1995). "Convergence Properties of the K-Means Algorithms". *Advances in Neural Information Processing Systems (NeurIPS)*, 7, 585-592.

5. **Vattani, A.** (2011). "K-means Requires Exponentially Many Iterations Even in the Plane". *Discrete & Computational Geometry*, 45(4), 596-616.

### Scalability and Variants

6. **Sculley, D.** (2010). "Web-Scale K-Means Clustering". *Proceedings of the 19th International Conference on World Wide Web*, 1177-1178.

7. **Hamerly, G., & Elkan, C.** (2002). "Alternatives to the k-means Algorithm that Find Better Clusterings". *Proceedings of the Eleventh International Conference on Information and Knowledge Management*, 600-607.

### Textbooks and Surveys

8. **Bishop, C. M.** (2006). *Pattern Recognition and Machine Learning*. Springer. (Chapter 9: Mixture Models and EM)

9. **Murphy, K. P.** (2012). *Machine Learning: A Probabilistic Perspective*. MIT Press. (Chapter 11: Mixture Models and the EM Algorithm)

10. **Jain, A. K.** (2010). "Data Clustering: 50 Years Beyond K-Means". *Pattern Recognition Letters*, 31(8), 651-666.

---

## Summary

**K-Means clustering** is a foundational unsupervised learning algorithm with:

✓ **Advantages**:
  - Simple, interpretable
  - Computationally efficient: O(nKdt)
  - Scales to large datasets
  - Works well for spherical, well-separated clusters

✗ **Limitations**:
  - Requires specifying K
  - Sensitive to initialization
  - Assumes spherical clusters
  - Finds only local minima

**Best Practices**:
1. Use k-means++ initialization
2. Run multiple times (n_init=10)
3. Validate with elbow method and silhouette scores
4. Consider alternatives for non-spherical data
5. Standardize features before clustering

**When to use K-Means**:
- Large datasets requiring efficiency
- Approximately spherical clusters
- Known or estimable number of clusters
- Quick exploratory analysis

**When to avoid K-Means**:
- Non-convex cluster shapes
- Hierarchical relationships important
- Variable cluster densities
- Need probabilistic assignments → Use GMM instead

---